# Google One AI Kit — 01. ComfyUI 云端生图工作台（L4 GPU）

> **运行前检查**：
> 1. 顶部菜单：**Runtime -> Change runtime type -> L4 GPU**（24GB 显存，约 1.7 单元/小时）。
> 2. 确认已完成 `00_setup_and_download_models.ipynb` 并将底模刷盘写入 Google Drive。
> 3. **收工必做**：画完图后立刻点 **Runtime -> Disconnect and delete runtime**，避免闲置按分钟扣点数！（所有产出图片会实时保存在 Drive `GoogleOneAI_Studio/outputs/`）

In [ ]:
# @title 1. 检查 GPU 显存、挂载 Google Drive 并校验底模
import os
import subprocess
import torch
from google.colab import drive

subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError("❌ 未检测到 GPU！请点击：Runtime -> Change runtime type -> L4 GPU。")

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
gpu_name = torch.cuda.get_device_name(0)
print(f"\n✅ 当前 GPU: {gpu_name} | 显存: {vram_gb:.1f} GB")
if vram_gb < 15.5:
    print("⚠️ 显存低于 16 GB，运行 FLUX.1-dev FP8 可能会爆显存，建议使用 L4 GPU (24 GB)。")

drive.mount("/content/drive")

STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
REQUIRED_FILES = [
    f"{STUDIO_DIR}/models/checkpoints/flux1-dev-fp8.safetensors",
    f"{STUDIO_DIR}/models/style_models/flux1-redux-dev.safetensors",
    f"{STUDIO_DIR}/models/clip_vision/sigclip_vision_patch14_384.safetensors",
]
for path in REQUIRED_FILES:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ 缺少模型文件: {path}\n"
            "请先使用 CPU 运行时完整执行 00_setup_and_download_models.ipynb。"
        )
    print(f"  [OK] {os.path.basename(path)} ({os.path.getsize(path)/(1024**3):.2f} GB)")

In [ ]:
# @title 2. 安装 ComfyUI 与 ComfyUI-Manager（装在 Colab 临时盘，启动最快）
import os
import subprocess
import sys

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR],
        check=True,
    )
else:
    print("ComfyUI 已存在，跳过 clone")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY_DIR}/requirements.txt"], check=True)

MANAGER_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"
if not os.path.exists(MANAGER_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ltdrdata/ComfyUI-Manager.git", MANAGER_DIR],
        check=True,
    )
mgr_req = f"{MANAGER_DIR}/requirements.txt"
if os.path.exists(mgr_req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", mgr_req], check=True)

print("✅ ComfyUI 与 Manager 准备就绪")

In [ ]:
# @title 3. 挂载 Drive 模型路径，并同步预设工作流与参考图到 ComfyUI
import glob
import os
import shutil

STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
COMFY_DIR = "/content/ComfyUI"

extra_yaml = f"""google_one_drive:
    base_path: {STUDIO_DIR}/models/
    checkpoints: checkpoints/
    loras: loras/
    style_models: style_models/
    clip_vision: clip_vision/
    diffusion_models: diffusion_models/
    text_encoders: text_encoders/
    vae: vae/
"""
with open(f"{COMFY_DIR}/extra_model_paths.yaml", "w", encoding="utf-8") as f:
    f.write(extra_yaml)

wf_dst = f"{COMFY_DIR}/user/default/workflows"
os.makedirs(wf_dst, exist_ok=True)
wf_files = sorted(glob.glob(f"{STUDIO_DIR}/workflows/*.json"))
for src in wf_files:
    shutil.copy2(src, os.path.join(wf_dst, os.path.basename(src)))

in_dst = f"{COMFY_DIR}/input"
os.makedirs(in_dst, exist_ok=True)
ref_files = [
    p for p in sorted(glob.glob(f"{STUDIO_DIR}/input_refs/*"))
    if os.path.isfile(p)
]
for src in ref_files:
    shutil.copy2(src, os.path.join(in_dst, os.path.basename(src)))

os.makedirs(f"{STUDIO_DIR}/outputs", exist_ok=True)
loras = sorted(glob.glob(f"{STUDIO_DIR}/models/loras/*.safetensors"))

print("✅ extra_model_paths.yaml 已写入")
print(f"✅ 已载入工作流 ({len(wf_files)}): {[os.path.basename(p) for p in wf_files]}")
print(f"✅ 已载入参考图 ({len(ref_files)}): {[os.path.basename(p) for p in ref_files]}")
print(f"ℹ️ 现有 LoRA 模型 ({len(loras)}): {[os.path.basename(p) for p in loras]}")

In [ ]:
# @title 4. 启动 ComfyUI 服务并建立 Cloudflare 公网隧道（点击输出 URL 打开界面）
import os
import re
import subprocess
import time
import urllib.request

COMFY_DIR = "/content/ComfyUI"
OUTPUT_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio/outputs"
CLOUDFLARED = "/content/cloudflared"
LOG_PATH = "/content/comfyui.log"

if not os.path.exists(CLOUDFLARED):
    print("⬇️ 下载 cloudflared...")
    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", CLOUDFLARED,
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

log_file = open(LOG_PATH, "w", encoding="utf-8")
comfy_cmd = [
    "python3", "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
    "--enable-cors-header",
    "--output-directory", OUTPUT_DIR,
]
comfy_proc = subprocess.Popen(comfy_cmd, cwd=COMFY_DIR, stdout=log_file, stderr=subprocess.STDOUT)

print("⏳ 正在等待 ComfyUI 就绪（端口 8188）...")
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH, "r", encoding="utf-8", errors="replace") as rf:
            tail = "".join(rf.readlines()[-60:])
        raise RuntimeError(f"❌ ComfyUI 提前退出（code={comfy_proc.returncode}），日志末尾:\n{tail}")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    comfy_proc.terminate()
    raise TimeoutError("❌ ComfyUI 180 秒内未就绪，请检查 /content/comfyui.log")

print("✅ ComfyUI 已就绪！正在建立 Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
try:
    for line in tunnel_proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if match and not tunnel_url:
            tunnel_url = match.group(0)
            print("=" * 72)
            print(f"🚀 ComfyUI 已上线，点击打开: {tunnel_url}")
            print("📁 预设工作流：左侧栏 Workflows -> 02_flux_redux_style.json（或 01_flux_base_lora.json）")
            print(f"💾 生成图片会自动保存到 Google Drive: {OUTPUT_DIR}")
            print("⚠️ 用完务必点本单元格左侧停止按钮，并执行 Runtime -> Disconnect and delete runtime！")
            print("=" * 72)
        if comfy_proc.poll() is not None:
            print("⚠️ ComfyUI 进程已停止，正在关闭隧道...")
            break
except KeyboardInterrupt:
    print("\n🛑 收到停止信号，正在关闭 ComfyUI 与隧道...")
finally:
    tunnel_proc.terminate()
    comfy_proc.terminate()
    log_file.close()
    print("✅ 服务已停止。记得点击顶部 Runtime -> Disconnect and delete runtime 释放 GPU！")